In [1]:
import pandas as pd

df = pd.read_csv("Superstore.csv",encoding = 'Latin-1')
#print(df.head())

In [11]:
def row_to_chunks(row):
    return [", ".join([f"{col}: {row[col]}" for col in row.index])]

chunks = []

for _, row in df.iterrows():
    chunks.extend(row_to_chunks(row))

In [3]:
from sentence_transformers import SentenceTransformer

c:\Users\divya\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = embedding_model.encode(chunks)

print(len(embeddings))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3424.38it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


29982


In [12]:
import numpy as np
import faiss

embeddings = np.array(embeddings).astype('float32')

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

In [13]:
import requests

def ask_llm(prompt):
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "mistral",
            "prompt": prompt,
            "stream": False
        }
    )
    return response.json()["response"]

In [14]:
import json

def parse_query(query):
    
    prompt = f"""
    You are a data assistant.

    Extract details from the query and return JSON with:
    - operation (sum, average, count, none)
    - column (column name exactly as in dataset)
    - filters (dictionary)
    - group_by (column if any)

    Query: {query}
    """
    
    response = ask_llm(prompt)
    
    try:
        return json.loads(response)
    except:
        return None

In [15]:
def execute_query(parsed, df):
    
    if not parsed:
        return None
    
    temp_df = df.copy()
    
    col = parsed.get("column")
    op = parsed.get("operation")
    
    # Apply filters
    if parsed.get("filters"):
        for key, val in parsed["filters"].items():
            if key in temp_df.columns:
                temp_df = temp_df[temp_df[key].astype(str).str.lower() == str(val).lower()]
    
    # Group by
    if parsed.get("group_by") and col in temp_df.columns:
        return temp_df.groupby(parsed["group_by"])[col].sum()
    
    # Operations
    if col in temp_df.columns:
        if op == "sum":
            return temp_df[col].sum()
        
        elif op == "average":
            return temp_df[col].mean()
    
    if op == "count":
        return len(temp_df)
    
    return None

In [16]:
def handle_insight_query(query):
    
    query_embedding = embedding_model.encode([query])
    query_embedding = np.array(query_embedding).astype('float32')
    
    k = 3
    distances, indices = index.search(query_embedding, k)
    
    retrieved_chunks = [chunks[i] for i in indices[0]]
    
    context = "\n".join(retrieved_chunks)
    
    prompt = f"""
    You are a senior data analyst.

    Analyze the data and provide EXACTLY 3 insights.

    Rules:
    - Keep each insight short
    - Focus on business impact

    Data:
    {context}

    Question: {query}
    """
    
    return ask_llm(prompt)

In [22]:
def handle_insight_query(query):
    
    query_embedding = embedding_model.encode([query])
    query_embedding = np.array(query_embedding).astype('float32')
    
    k = min(3, len(chunks))
    
    distances, indices = index.search(query_embedding, k)
    
    retrieved_chunks = [
        chunks[i] for i in indices[0]
        if i >= 0 and i < len(chunks)
    ]
    
    context = "\n".join(retrieved_chunks)
    
    prompt = f"""
    You are a senior data analyst.

    Analyze the data and provide the exact answer. Do not give steps required for getting the answer.

    Data:
    {context}

    Question: {query}
    """
    
    return ask_llm(prompt)

In [23]:
while True:
    
    print("\n" + "="*50)
    
    query = input("👉 Ask your question (type 'exit' to quit): ")
    
    if query.lower() == "exit":
        print("Goodbye 👋")
        break
    
    # 🔥 Smart parsing
    parsed = parse_query(query)
    
    result = execute_query(parsed, df)
    
    if result is not None:
        print("\n📊 Data Result:\n")
        print(result)
    
    else:
        answer = handle_insight_query(query)
        
        print("\n🤖 AI Insights:\n")
        print(answer)
    
    print("\n" + "="*50)



🤖 AI Insights:

 Based on the provided data, the total sales by region are as follows:

1. North: $8000 + $6500 + $5200 = $20,700
2. South: $9800 + $8400 = $18,200
3. East: $5600 + $6900 + $7000 + $7100 = $27,300
4. West: $7200 + $8800 + $6600 = $22,600

Total sales across all regions: $20,700 (North) + $18,200 (South) + $27,300 (East) + $22,600 (West) = $90,400.


Goodbye 👋
